# Module 7: Alerts - Close the Loop

## Learning Objectives
- Create email alerts triggered by DQ failures
- Build scheduled tasks that sweep for quality issues
- Combine tasks + alerts for end-to-end automation
- Understand the full DQ monitoring lifecycle

## Key Concept: The Complete Lifecycle

```
Data Lands --> DMFs Run --> Expectations Evaluated --> Alert Fires --> Team Acts
   (auto)      (auto)          (auto)                  (auto)        (human)
```

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes
> **Reference:** [STUDENT_GUIDE.md](../guide/STUDENT_GUIDE.md) — Module 7 covers the alert lifecycle and scheduled sweep pattern.


> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## 7a. Create an Alert

> **Business Value:** Without alerts, DQ failures sit unnoticed until a business user reports incorrect data -- often days or weeks later. Alerts turn hours of damage into minutes.

> **DQ Domain:** All (alerting infrastructure)

> **ACTION REQUIRED:** Replace `'your.email@company.com'` in the cell below with the email address you used in Module 0's notification integration setup.

In [ ]:
CREATE OR REPLACE ALERT CORP_DWH.DQ.ALERT_DQ_FAILURES
    WAREHOUSE = DQ_LAB_WH
    SCHEDULE = '5 MINUTES'
IF (EXISTS (
    SELECT 1
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER',
        REF_ENTITY_DOMAIN => 'TABLE'
    ))
    WHERE EXPECTATION_RESULT = 'NOT_MET'
      AND MEASUREMENT_TIME > DATEADD(MINUTE, -10, CURRENT_TIMESTAMP())
))
THEN
    CALL SYSTEM$SEND_EMAIL(
        'CORP_DQ_ALERTS',
        'your.email@company.com',
        'DQ ALERT: Gold Layer Quality Failure',
        'One or more DQ expectations FAILED on CORP_DWH.GOLD.DIM_CUSTOMER. Review immediately.'
    );

---
## 7b. Resume the Alert

> **What this does:** Activates the alert so it starts checking for DQ failures on its schedule.


In [ ]:
ALTER ALERT CORP_DWH.DQ.ALERT_DQ_FAILURES RESUME;

---
## 7c. Build a DQ Sweep

> **Business Value:** The sweep log creates an auditable history. When regulators ask 'how do you ensure data quality?', you show continuous monitoring records, not one-time checks.

> **DQ Domain:** All (scheduled monitoring)

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.DQ.DQ_SWEEP_LOG (
    SWEEP_ID NUMBER AUTOINCREMENT,
    SWEEP_TIME TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    TABLE_NAME STRING,
    TOTAL_EXPECTATIONS NUMBER,
    PASSED NUMBER,
    FAILED NUMBER
);

> **What this does:** Creates a stored procedure that sweeps DQ monitoring results for the last hour and logs pass/fail counts into the sweep log table.


In [ ]:
CREATE OR REPLACE PROCEDURE CORP_DWH.DQ.RUN_DQ_SWEEP()
RETURNS STRING
LANGUAGE SQL
AS
DECLARE
    tables_checked NUMBER DEFAULT 0;
BEGIN
    INSERT INTO CORP_DWH.DQ.DQ_SWEEP_LOG (TABLE_NAME, TOTAL_EXPECTATIONS, PASSED, FAILED)
    SELECT
        'CORP_DWH.GOLD.DIM_CUSTOMER',
        COUNT(*),
        COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END),
        COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END)
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER',
        REF_ENTITY_DOMAIN => 'TABLE'
    ))
    WHERE EXPECTATION_NAME IS NOT NULL
      AND MEASUREMENT_TIME > DATEADD(HOUR, -1, CURRENT_TIMESTAMP());

    tables_checked := tables_checked + 1;
    RETURN 'DQ sweep complete. Tables checked: ' || :tables_checked;
END;

---
## 7d. Create Scheduled Task

> **DQ Domain:** Freshness + Volume (periodic checks)

In [ ]:
CREATE OR REPLACE TASK CORP_DWH.DQ.TASK_HOURLY_DQ_SWEEP
    WAREHOUSE = DQ_LAB_WH
    SCHEDULE = '60 MINUTES'
AS
    CALL CORP_DWH.DQ.RUN_DQ_SWEEP();

ALTER TASK CORP_DWH.DQ.TASK_HOURLY_DQ_SWEEP RESUME;

---
## 7e. Test: Run the Sweep Manually

> **What this does:** Executes the RUN_DQ_SWEEP procedure and shows the result.


In [ ]:
CALL CORP_DWH.DQ.RUN_DQ_SWEEP();

> **What this does:** Displays the most recent sweep log entries to confirm the procedure ran successfully.


In [ ]:
SELECT * FROM CORP_DWH.DQ.DQ_SWEEP_LOG ORDER BY SWEEP_TIME DESC LIMIT 5;

---
## Checkpoint: Full Lifecycle Verification

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("FINAL CHECKPOINT: Complete DQ Lifecycle Verification")
print("=" * 60)
passed = 0
total = 5

# 1. Alert exists
try:
    session.sql("DESCRIBE ALERT CORP_DWH.DQ.ALERT_DQ_FAILURES").collect()
    print("  [PASS] Alert ALERT_DQ_FAILURES exists")
    passed += 1
except:
    print("  [FAIL] Alert not found")

# 2. Task exists
try:
    session.sql("DESCRIBE TASK CORP_DWH.DQ.TASK_HOURLY_DQ_SWEEP").collect()
    print("  [PASS] Task TASK_HOURLY_DQ_SWEEP exists")
    passed += 1
except:
    print("  [FAIL] Task not found")

# 3. Sweep log has entries
log_count = session.sql("SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.DQ_SWEEP_LOG").collect()[0]['CNT']
if log_count > 0:
    print(f"  [PASS] DQ_SWEEP_LOG has {log_count} entries")
    passed += 1
else:
    print("  [FAIL] DQ_SWEEP_LOG is empty - run CALL CORP_DWH.DQ.RUN_DQ_SWEEP()")

# 4. Procedure works
try:
    result = session.sql("CALL CORP_DWH.DQ.RUN_DQ_SWEEP()").collect()[0][0]
    print(f"  [PASS] RUN_DQ_SWEEP() returned: {result}")
    passed += 1
except Exception as e:
    print(f"  [FAIL] Procedure error: {str(e)[:80]}")

# 5. Rules catalog populated
rules = session.sql("SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG WHERE DMF_NAME IS NOT NULL").collect()[0]['CNT']
if rules > 0:
    print(f"  [PASS] Rules Catalog: {rules} rules with provisioned DMFs")
    passed += 1
else:
    print("  [WARN] No provisioned rules found")
    passed += 1

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 60)
print()
print("CONGRATULATIONS! You have completed the full DQ Monitoring lifecycle:")
print("  1. System DMFs at Raw layer")
print("  2. Custom DMFs at Silver layer")
print("  3. Rules Catalog at Gold layer")
print("  4. Expectations for pass/fail verdicts")
print("  5. AI-powered anomaly detection and rule suggestions")
print("  6. Governance integration (tags, classification, lineage)")
print("  7. Automated alerts and scheduled sweeps")
print("=" * 60)

---
## Final Quiz: Full Lifecycle

**Q1:** Draw the complete DQ lifecycle from data landing to alert firing (mentally or on paper). How many automated steps are there before a human needs to act?

**Q2:** You're onboarding a new data source (a third-party API feed). List the steps to add DQ monitoring using what you learned today.

**Q3:** What's the difference between an Alert and a Task in the context of DQ?

**Q4:** Your manager asks: "What's our overall data quality score?" How would you answer using what we built?

> **What this does:** Reveals quiz answers. Try answering first!


In [ ]:
print("""
FINAL QUIZ ANSWERS
==================

Q1: Complete lifecycle:
    1. Data lands in RAW table (auto - Snowpipe/COPY)
    2. DATA_METRIC_SCHEDULE triggers DMFs (auto - serverless)
    3. DMFs compute metric values (auto - serverless)
    4. Expectations evaluate pass/fail (auto - built-in)
    5. Alert condition checks for failures (auto - 5-min schedule)
    6. Email notification sent (auto - SYSTEM$SEND_EMAIL)
    7. Human reviews and remediates (HUMAN ACTION)
    = 6 automated steps before human intervention.

Q2: Steps to onboard a new source:
    1. Create staging table in RAW schema
    2. Attach system DMFs (ROW_COUNT, FRESHNESS, NULL_COUNT on key columns)
    3. Set DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES'
    4. Add expectations (VALUE = 0 for nulls, VALUE <= threshold for freshness)
    5. Add business rules to RULES_CATALOG
    6. CALL PROVISION_DMFS_FROM_CATALOG()
    7. (Optional) Use Cortex AI to suggest additional rules
    8. Add alert condition for the new table

Q3: Alert vs Task:
    - ALERT: Reactive. Checks a CONDITION and fires an ACTION only when true.
      Best for: "Tell me when something goes wrong."
    - TASK: Proactive. Runs on a SCHEDULE regardless of conditions.
      Best for: "Collect DQ metrics every hour and log them."
    Use both together: Task sweeps and logs; Alert fires when thresholds breach.

Q4: DQ Scorecard approach:
    - Query DQ_SWEEP_LOG for latest sweep
    - Calculate: (PASSED / TOTAL_EXPECTATIONS) * 100 = DQ Score %
    - Break down by table, severity, and team (from RULES_CATALOG)
    - Present: "Gold layer is at 78% quality. 4 CRITICAL expectations failing.
      Owner: Compliance Team and HR. Root cause: invalid National IDs from Gov Portal feed."
""")

---
## Cleanup (Optional)

Suspend alert and task to avoid ongoing compute:

In [ ]:
ALTER ALERT CORP_DWH.DQ.ALERT_DQ_FAILURES SUSPEND;
ALTER TASK CORP_DWH.DQ.TASK_HOURLY_DQ_SWEEP SUSPEND;

---
## Summary

You have built the full DQ monitoring lifecycle: detect, evaluate, alert, and sweep.

### What You Built in This Module
- **Email Alert** that fires automatically on DQ failures
- **Scheduled Task** running hourly DQ sweeps
- **DQ Sweep Log** providing auditable monitoring history
- **Complete lifecycle** from data landing to human action

---

**Next:** Open `8A_DASHBOARD_NATIVE` (SQL-only), `8B_DASHBOARD_PYTHON` (matplotlib), or `8C_DASHBOARD_STREAMLIT` (Streamlit app) to build a DQ dashboard. Choose one variant based on your preference.